# Cloud Cost Analysis

This notebook explores multi-cloud pricing data and identifies cost optimization opportunities.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Add src to path
sys.path.insert(0, str(Path().absolute().parent / "src"))

from cost_model import CostModel
from fetch_pricing import fetch_all_pricing, normalize_pricing_data


## Load Pricing Data


In [ ]:
# Load normalized data
with open("../data/normalized.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Loaded {len(df)} pricing SKUs")
print(f"Providers: {df['provider'].unique()}")
print(f"\nData shape: {df.shape}")
df.head()


## Compute Instance Comparison


In [ ]:
# Filter compute instances
compute_df = df[df['price_per_hour'].notna()].copy()

print(f"Compute instances: {len(compute_df)}")
print("\nBy Provider:")
print(compute_df.groupby('provider').agg({
    'price_per_hour': ['mean', 'min', 'max'],
    'vcpu': 'mean',
    'memory_gb': 'mean'
}))


## 5-Year TCO Analysis


In [ ]:
# Initialize cost model
model = CostModel("../data/normalized.json")

# Compare providers for equivalent instances
comparisons = model.compare_providers(
    "2vcpu-8gb",
    "us-east-1",
    hours_per_month=730,
    years=5
)

tco_df = pd.DataFrame(comparisons)
print("5-Year TCO Comparison (730 hours/month):")
print(tco_df[["provider", "total_cost_5yr", "hourly_price"]].to_string(index=False))


## Cost Savings Analysis


In [ ]:
# Calculate potential savings
if len(tco_df) > 0:
    cheapest = tco_df.loc[tco_df['total_cost_5yr'].idxmin()]
    most_expensive = tco_df.loc[tco_df['total_cost_5yr'].idxmax()]
    
    savings = ((most_expensive['total_cost_5yr'] - cheapest['total_cost_5yr']) / 
               most_expensive['total_cost_5yr'] * 100)
    
    print(f"💰 Cost Optimization Opportunity:")
    print(f"   Cheapest: {cheapest['provider'].upper()} - ${cheapest['total_cost_5yr']:,.2f}")
    print(f"   Most Expensive: {most_expensive['provider'].upper()} - ${most_expensive['total_cost_5yr']:,.2f}")
    print(f"   Potential Savings: {savings:.1f}%")
    print(f"   Absolute Savings: ${most_expensive['total_cost_5yr'] - cheapest['total_cost_5yr']:,.2f}")
